# 04 — Construção dos modelos

**Enunciado etapa 4** \
Construção dos modelos: Faça a classificação usando diferentes modelos. Faça a escolha dos hiperparâmetros usando validação cruzada.

**Modelos**

| Modelo | Família                         | Escolha dos hiperparâmetros |
|---|---------------------------------|-----------------------------|
| Dummy | referência (não aprende nada)   | —                           |
| Regressão Logística | linear              | Fixos, para referência      |
| Regressão Logística | linear                          | `GridSearchCV`              |
| KNN | baseado em distância            | `GridSearchCV`              |
| Árvore de Decisão | árvore única                    | `GridSearchCV`              |
| Random Forest | ensemble de árvores (*bagging*) | Optuna                      |
| XGBoost | ensemble de árvores (*boosting*) | Optuna                      |
| LightGBM | ensemble de árvores (*boosting*) | Optuna                      |

**Saídas deste notebook**:
- `results/hiperparametros.json` — melhores hiperparâmetros de cada modelo e conjunto
- `results/cv_original.csv` — métricas de cada fold na comparação final
- `results/oof_original.csv` — predições fora da amostra no treino (usadas na escolha do limiar, notebook 06)
- `models/*__original.joblib` — pipelines ajustados no treino completo

In [10]:
# Imports
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## 1. Protocolo de avaliação
**[IA]** - Tive auxílio da IA para organizar a aplicação da validação cruzada e a exibição dos resultados das avaliações em funções reutilizáveis para cada modelo, afim de tornar o notebook mais limpo

**Validação cruzada**
- `StratifiedKFold` com 5 folds e semente 42. Todas as configurações de todos os modelos são avaliadas nesses mesmos 5 folds.

**Métricas**
- **Average Precision (AP)**: métrica principal, usada para escolher os hiperparâmetros. Resume a curva precisão–sensibilidade, ou seja, o quanto o modelo coloca os clientes que vão cancelar no topo do ranking. Um modelo aleatório tem AP igual à taxa de churn (~0,265).
- **ROC-AUC**: também mede a ordenação, mas é menos sensível ao desbalanceamento.
- **Precisão, sensibilidade e F1**: dependem do limiar de decisão. Aqui uso o limiar padrão de 0,5.

In [11]:
SEMENTE = 42

# Os mesmos 5 folds do notebook 03
CV_BUSCA = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

# Métricas da classe churn. Precisão, sensibilidade e F1 usam o limiar padrão de 0,5;
# zero_division=0 evita avisos quando o modelo não prevê nenhum churn (ex.: o Dummy)
METRICAS = {
    "ap": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, zero_division=0),
    "precisao": make_scorer(precision_score, zero_division=0),
    "sensibilidade": make_scorer(recall_score, zero_division=0),
}
METRICA_BUSCA = "ap"


def avaliar(modelo, X, y, cv=CV_BUSCA, treino=False):
    # Métricas de cada fold. Com treino=True também devolve as métricas no treino do fold
    # (colunas "<métrica>_treino"), para medir sobreajuste
    r = cross_validate(modelo, X, y, cv=cv, scoring=METRICAS, n_jobs=-1, return_train_score=treino)
    res = pd.DataFrame({m: r[f"test_{m}"] for m in METRICAS})
    if treino:
        for m in METRICAS:
            res[f"{m}_treino"] = r[f"train_{m}"]
    return res.rename_axis("fold").reset_index()


def resumir(resultados, por=("modelo", "conjunto"), metricas=tuple(METRICAS)):
    # Média ± desvio padrão entre os folds
    g = resultados.groupby(list(por), sort=False)[list(metricas)]
    media, desvio = g.mean(), g.std()
    return media.round(3).astype(str) + " ± " + desvio.round(3).astype(str)

## 2. Leitura dos dados

Uso o treino com as variáveis criadas no notebook 03 (`train_fe.csv`) e os dois conjuntos de variáveis escolhidos lá (`feature_set.json`):
- **`base`**: as 19 variáveis originais
- **`engenharia`**: `base` + as 5 variáveis criadas

In [12]:
# Treino com as variáveis criadas no notebook 03
train = pd.read_csv("../data/processed/train_fe.csv")
TARGET = "Churn"
X = train.drop(columns=["customerID", TARGET])
y = train[TARGET]

# Conjuntos de variáveis escolhidos no notebook 03
with open("../data/processed/feature_set.json", encoding="utf-8") as f:
    CONJUNTOS = json.load(f)

print(X.shape, f"taxa de churn no treino = {y.mean():.1%}")
X.head()

(5634, 24) taxa de churn no treino = 26.5%


,is_male,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,n_servicos_adicionais,servicos_suporte,log_tenure,variacao_mensalidade,mensal_x_fibra
0,1,0,0,0,35,0,0,DSL,0,0,...,Month-to-month,0,Electronic check,49.20,1701.65,3,0,3.583519,0.581429,0
1,1,0,1,1,15,1,0,Fiber optic,1,0,...,Month-to-month,0,Mailed check,75.10,1151.55,1,1,2.772589,-1.670000,1
2,1,0,1,1,13,0,0,DSL,1,1,...,Two year,0,Mailed check,40.55,590.35,3,2,2.639057,-4.861538,0
3,0,0,1,0,26,1,0,DSL,0,1,...,Two year,1,Credit card (automatic),73.50,1905.70,4,0,3.295837,0.203846,0
4,1,0,1,1,1,1,0,DSL,0,0,...,Month-to-month,0,Electronic check,44.55,44.55,0,0,0.693147,0.000000,0


In [13]:
# Número de variáveis de cada tipo em cada conjunto
pd.DataFrame({nome: {tipo: len(cols) for tipo, cols in c.items()} for nome, c in CONJUNTOS.items()}).T.assign(
    total=lambda d: d.sum(axis=1))

,numericas,binarias,categoricas,total
base,3,13,3,19
engenharia,7,14,3,24


In [14]:
# Tamanho de cada fold de validação da busca e quantos churns ele tem
pd.DataFrame([{"n": len(val), "churns": int(y.iloc[val].sum()), "taxa de churn": y.iloc[val].mean()}
              for _, val in CV_BUSCA.split(X, y)]).rename_axis("fold").round(3)

,n,churns,taxa de churn
fold,,,
0,1127,299,0.265
1,1127,299,0.265
2,1127,299,0.265
3,1127,299,0.265
4,1126,299,0.266


## 3. Pré-processamento e pipelines

O pré-processamento foi definido no notebook 03:
- **one-hot** com `drop="first"` em `InternetService`, `Contract` e `PaymentMethod`
- **padronização** (`StandardScaler`) das numéricas
- binárias sem alteração

Cada modelo é colocado em um `Pipeline` junto com o pré-processamento. Assim, na validação cruzada, o one-hot e a padronização são ajustados só no treino de cada fold, sem vazamento de informação do fold de validação.

In [15]:
def fazer_preprocessador(numericas, binarias, categoricas, escalar=True):
    # Mesmo pré-processamento do notebook 03: one-hot nas categóricas, padronização
    # nas numéricas e binárias sem alteração. Colunas fora do conjunto são descartadas
    # (remainder="drop"), então o conjunto "base" simplesmente ignora as variáveis criadas
    return ColumnTransformer([
        ("num", StandardScaler() if escalar else "passthrough", numericas),
        ("bin", "passthrough", binarias),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categoricas),
    ], verbose_feature_names_out=False)


def montar_pipeline(modelo, conjunto, escalar=True):
    # Pré-processamento + modelo: na validação cruzada, tudo é ajustado só no treino de cada fold
    return Pipeline([("prep", fazer_preprocessador(**conjunto, escalar=escalar)), ("modelo", modelo)])

In [16]:
# Colunas que cada conjunto gera depois do pré-processamento
for nome, c in CONJUNTOS.items():
    colunas = fazer_preprocessador(**c).fit(X).get_feature_names_out()
    print(f"{nome}: {len(colunas)} colunas\n  {list(colunas)}\n")

base: 23 colunas
  ['tenure', 'MonthlyCharges', 'TotalCharges', 'is_male', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

engenharia: 28 colunas
  ['tenure', 'MonthlyCharges', 'TotalCharges', 'n_servicos_adicionais', 'servicos_suporte', 'log_tenure', 'variacao_mensalidade', 'is_male', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'mensal_x_fibra', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMe

In [17]:
# Exemplo de pipeline completo
montar_pipeline(LogisticRegression(max_iter=5000), CONJUNTOS["engenharia"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}_

## 4. Modelos de referência

Antes de ajustar hiperparâmetros, avalio dois modelos que servem de referência:

- **Dummy** (`strategy="prior"`): prevê para todo cliente a mesma probabilidade, a taxa de churn do treino. É o piso: a AP dele é a própria taxa de churn (~0,265), a ROC-AUC é 0,5 e, como a probabilidade prevista (0,265) fica abaixo do limiar de 0,5, ele nunca prevê churn (precisão, sensibilidade e F1 iguais a 0). Qualquer modelo útil precisa ficar bem acima dele.
- **Regressão Logística com hiperparâmetros padrão** (`C=1`, penalidade L2): é o mesmo modelo usado no notebook 03, nos mesmos folds, então deve reproduzir a AP de lá (0,661 no `base` e 0,671 no `engenharia`). Também serve de referência para a busca: um modelo ajustado só vale a pena se superar essa configuração padrão.

In [18]:
REFERENCIAS = {
    "Dummy": DummyClassifier(strategy="prior"),
    "LogReg (padrão)": LogisticRegression(max_iter=5000),
}

linhas = []
for nome_mod, modelo in REFERENCIAS.items():
    for nome_conj, c in CONJUNTOS.items():
        res = avaliar(montar_pipeline(modelo, c), X, y, cv=CV_BUSCA, treino=True)
        linhas.append(res.assign(modelo=nome_mod, conjunto=nome_conj))
res_ref = pd.concat(linhas, ignore_index=True)
resumir(res_ref)

ap        roc_auc             f1  \
modelo          conjunto                                                  
Dummy           base          0.265 ± 0.0      0.5 ± 0.0      0.0 ± 0.0   
                engenharia    0.265 ± 0.0      0.5 ± 0.0      0.0 ± 0.0   
LogReg (padrão) base        0.661 ± 0.022  0.846 ± 0.014  0.593 ± 0.034   
                engenharia  0.671 ± 0.017  0.849 ± 0.013  0.597 ± 0.032   

                                 precisao  sensibilidade  
modelo          conjunto                                  
Dummy           base            0.0 ± 0.0      0.0 ± 0.0  
                engenharia      0.0 ± 0.0      0.0 ± 0.0  
LogReg (padrão) base        0.654 ± 0.031  0.543 ± 0.046  
                engenharia  0.673 ± 0.032  0.538 ± 0.042

In [19]:
# AP no treino x AP na validação (média dos folds): uma diferença grande indicaria sobreajuste
(res_ref.groupby(["modelo", "conjunto"], sort=False)[["ap_treino", "ap"]].mean()
 .rename(columns={"ap_treino": "AP treino", "ap": "AP validação"})
 .assign(diferença=lambda d: d["AP treino"] - d["AP validação"])
 .round(3))

AP treino  AP validação  diferença
modelo          conjunto                                      
Dummy           base            0.265         0.265     -0.000
                engenharia      0.265         0.265     -0.000
LogReg (padrão) base            0.667         0.661      0.006
                engenharia      0.680         0.671      0.009

### 5. Ajuste de hiperparâmetros
Escolhi os modelos: `Regressão Logística`, `KNN`, `Random Forest`, `XGBoost`, `LightGBM`

Para cada modelo, testo várias configurações de hiperparâmetros nos dois conjuntos de variáveis (`base` e `engenharia`) e escolho a de **maior AP média nos 5 folds**.

- Para **Regressão Logística e KNN** uso `GridSearchCV`: como têm poucos hiperparâmetros, dá para testar todas as combinações de uma grade e desenhar a curva de AP (treino × validação) em função de cada um.
- Para **Random Forest, XGBoost e LightGBM** uso Optuna: O Optuna testa um número fixo de configurações, escolhendo cada nova configuração com base nas que já tiveram bom desempenho. Os três modelos recebem o mesmo `N_TRIALS`.


In [ ]:
# Melhor configuração de cada modelo e conjunto: {(modelo, conjunto): {...}}
MELHORES = {}

def registrar(modelo, conjunto, params, ap, ap_std, ap_treino, n_config, na_borda):
    MELHORES[(modelo, conjunto)] = {"params": params, "ap": ap, "ap_std": ap_std, "ap_treino": ap_treino,
                                    "n_config": n_config, "na_borda": na_borda}


def mostrar_melhores(modelo):
    # Melhor configuração do modelo em cada conjunto (uma coluna por conjunto)
    def arredondar(v):
        return float(f"{v:.4g}") if isinstance(v, float) else v
    linhas = {}
    for (m, conj), v in MELHORES.items():
        if m == modelo:
            linhas[conj] = {"AP validação": f"{v['ap']:.3f} ± {v['ap_std']:.3f}",
                            "AP treino": round(v["ap_treino"], 3),
                            "treino − validação": round(v["ap_treino"] - v["ap"], 3),
                            "configurações testadas": v["n_config"],
                            "na borda": ", ".join(v["na_borda"]) or "—",
                            **{k: arredondar(p) for k, p in v["params"].items()}}
    return pd.DataFrame(linhas)


def na_borda_grade(melhor, grade):
    # Hiperparâmetros numéricos (com mais de 2 valores na grade) cujo melhor valor é o menor ou o maior testado
    return [k for k, valores in grade.items()
            if len(valores) > 2 and all(isinstance(v, (int, float)) for v in valores)
            and melhor[k] in (min(valores), max(valores))]


def busca_grid(nome, modelo, grade):
    # GridSearchCV nos dois conjuntos; devolve a AP (validação e treino) de todas as configurações
    tabelas = []
    for nome_conj, c in CONJUNTOS.items():
        busca = GridSearchCV(montar_pipeline(modelo, c), {f"modelo__{k}": v for k, v in grade.items()},
                             scoring="average_precision", cv=CV_BUSCA, n_jobs=-1,
                             return_train_score=True, refit=False)
        busca.fit(X, y)
        r = busca.cv_results_
        tab = pd.DataFrame(r["params"]).rename(columns=lambda k: k.removeprefix("modelo__"))
        tab = tab.assign(conjunto=nome_conj, ap=r["mean_test_score"], ap_std=r["std_test_score"],
                         ap_treino=r["mean_train_score"])
        i = busca.best_index_
        melhor = {k.removeprefix("modelo__"): v for k, v in busca.best_params_.items()}
        registrar(nome, nome_conj, melhor, tab.loc[i, "ap"], tab.loc[i, "ap_std"], tab.loc[i, "ap_treino"],
                  len(tab), na_borda_grade(melhor, grade))
        tabelas.append(tab)
    return pd.concat(tabelas, ignore_index=True)


def curva_validacao(tab, x, cor, titulo, log=False):
    # AP de validação (linha cheia, faixa = ± 1 desvio entre folds) e de treino (tracejada) em função de x,
    # uma linha por valor de `cor` e um painel por conjunto. A linha pontilhada marca o melhor valor
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
    for ax, (nome_conj, t) in zip(axes, tab.groupby("conjunto", sort=False)):
        for (rotulo, g), cor_linha in zip(t.groupby(cor), sns.color_palette()):
            g = g.sort_values(x)
            ax.plot(g[x], g["ap"], marker="o", ms=3, color=cor_linha, label=f"{rotulo} (validação)")
            ax.fill_between(g[x], g["ap"] - g["ap_std"], g["ap"] + g["ap_std"], color=cor_linha, alpha=0.12)
            if g["ap_treino"].mean() < 0.999:  # AP de treino ~1 (KNN com weights="distance") achataria o gráfico
                ax.plot(g[x], g["ap_treino"], ls="--", color=cor_linha, label=f"{rotulo} (treino)")
        melhor = t.loc[t["ap"].idxmax()]
        ax.axvline(melhor[x], color="gray", ls=":", lw=1)
        ax.set(title=f"{titulo}: conjunto {nome_conj} (melhor AP = {melhor['ap']:.3f})", xlabel=x, ylabel="AP")
        if log:
            ax.set_xscale("log")
    axes[0].legend(title=cor, fontsize=7, title_fontsize=8)
    fig.tight_layout()
    plt.show()

#### 5.1 Regressão Logística

A regressão logística é regularizada: o ajuste penaliza o tamanho dos coeficientes, o que reduz o sobreajuste.
- **`C`**: inverso da força da regularização. `C` pequeno = penalização forte (coeficientes encolhidos em direção a 0, modelo mais simples); `C` grande = penalização fraca (quase a regressão logística sem regularização). Testo 11 valores em escala log, de 0,001 a 100.
- **`l1_ratio`**: tipo de penalização. `0` = **L2** (*ridge*), que encolhe todos os coeficientes; `1` = **L1** (*lasso*), que pode zerar coeficientes, fazendo uma seleção de variáveis automática.

Uso o *solver* `liblinear`, que aceita as duas penalizações e é rápido em bases pequenas.

In [ ]:
GRADE_LOGREG = {"C": np.logspace(-3, 2, 11).tolist(), "l1_ratio": [0, 1]}

tab_logreg = busca_grid("LogReg", LogisticRegression(solver="liblinear", max_iter=5000), GRADE_LOGREG)
tab_logreg.sort_values("ap", ascending=False).head(6).round(4)

In [ ]:
curva_validacao(tab_logreg, "C", "l1_ratio", "Regressão Logística", log=True)

In [ ]:
mostrar_melhores("LogReg")

#### 5.2 KNN

O KNN classifica cada cliente pela proporção de churn entre os `k` clientes mais parecidos do treino.
- **`n_neighbors` (k)**: número de vizinhos. `k` pequeno = fronteira muito irregular, sensível a ruído (sobreajuste); `k` grande = probabilidades mais suaves, mas que podem perder padrões locais.
- **`weights`**: `uniform` = todos os vizinhos têm o mesmo peso; `distance` = vizinhos mais próximos pesam mais.
- **`p`**: a distância usada. `1` = Manhattan (soma das diferenças absolutas); `2` = euclidiana.

Com `weights="distance"`, a AP no treino é praticamente 1: cada cliente do treino é vizinho de si mesmo, a distância 0, e recebe peso infinito. Por isso a AP de treino não serve para medir sobreajuste nessa configuração, e essas linhas não aparecem no gráfico.

In [ ]:
GRADE_KNN = {"n_neighbors": [5, 11, 21, 31, 41, 51, 61, 75, 91, 111, 131, 151, 201, 251, 301, 401],
             "weights": ["uniform", "distance"], "p": [1, 2]}

tab_knn = busca_grid("KNN", KNeighborsClassifier(), GRADE_KNN)
tab_knn.sort_values("ap", ascending=False).head(6).round(4)

In [ ]:
# Uma linha por combinação de weights e p
tab_knn["weights, p"] = tab_knn["weights"] + ", p=" + tab_knn["p"].astype(str)
curva_validacao(tab_knn, "n_neighbors", "weights, p", "KNN")

In [ ]:
mostrar_melhores("KNN")

#### Busca com Optuna

Nos três modelos de árvores, cada *trial* do Optuna:
1. sorteia uma configuração dentro do espaço de busca (função `espaco_*` de cada modelo);
2. avalia essa configuração nos mesmos 5 folds (AP média de validação);
3. informa o resultado ao amostrador TPE, que passa a sortear mais configurações nas regiões que foram bem.

O amostrador tem semente fixa, então a busca é reprodutível. Depois de cada busca mostro:
- o **histórico** da AP por trial (pontos) e a melhor AP encontrada até cada trial (linha), para ver se a busca estabilizou;
- a **importância de cada hiperparâmetro** (fANOVA): quanto da variação da AP entre os trials é explicada por cada hiperparâmetro.

Alguns hiperparâmetros têm um limite natural, que não dá para ampliar (por exemplo, `min_samples_leaf = 1` ou `max_depth = 1`). Quando o melhor valor fica nesse limite, a coluna "na borda" o aponta, mas não há como estender a busca.

In [ ]:
N_TRIALS = 100  # orçamento igual para Random Forest, XGBoost e LightGBM
optuna.logging.set_verbosity(optuna.logging.WARNING)


def na_borda_optuna(trial, margem=0.05):
    # Hiperparâmetros numéricos cujo melhor valor ficou nos 5% extremos do intervalo de busca
    # (em escala log quando o intervalo é log)
    borda = []
    for nome, d in trial.distributions.items():
        if isinstance(d, (optuna.distributions.FloatDistribution, optuna.distributions.IntDistribution)):
            v, lo, hi = trial.params[nome], d.low, d.high
            if d.log:
                v, lo, hi = np.log(v), np.log(lo), np.log(hi)
            posicao = (v - lo) / (hi - lo)
            if posicao < margem or posicao > 1 - margem:
                borda.append(nome)
    return borda


def busca_optuna(nome, modelo, espaco, n_trials=N_TRIALS):
    # Um estudo do Optuna por conjunto; objetivo = AP média de validação nos 5 folds
    estudos = {}
    for nome_conj, c in CONJUNTOS.items():
        def objetivo(trial):
            pipe = montar_pipeline(clone(modelo).set_params(**espaco(trial)), c)
            r = cross_validate(pipe, X, y, cv=CV_BUSCA, scoring="average_precision", n_jobs=-1,
                               return_train_score=True)
            trial.set_user_attr("ap_std", r["test_score"].std())
            trial.set_user_attr("ap_treino", r["train_score"].mean())
            return r["test_score"].mean()

        estudo = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEMENTE))
        estudo.optimize(objetivo, n_trials=n_trials)
        t = estudo.best_trial
        registrar(nome, nome_conj, t.params, t.value, t.user_attrs["ap_std"], t.user_attrs["ap_treino"],
                  n_trials, na_borda_optuna(t))
        estudos[nome_conj] = estudo
    return estudos


def mostrar_optuna(estudos, nome):
    # Histórico da busca e importância dos hiperparâmetros (fANOVA), nos dois conjuntos
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    piso = []
    for (nome_conj, estudo), cor in zip(estudos.items(), sns.color_palette()):
        df = estudo.trials_dataframe()
        axes[0].scatter(df["number"], df["value"], s=12, alpha=0.4, color=cor)
        axes[0].plot(df["number"], df["value"].cummax(), color=cor, label=f"{nome_conj} (melhor até o trial)")
        piso.append(df["value"].quantile(0.1))
    axes[0].set_ylim(bottom=min(piso) - 0.005)  # esconde os piores 10% dos trials para dar zoom no topo
    axes[0].set(title=f"{nome}: AP de validação por trial", xlabel="trial", ylabel="AP")
    axes[0].legend(fontsize=8)

    avaliador = optuna.importance.FanovaImportanceEvaluator(seed=SEMENTE)
    imp = pd.DataFrame({nome_conj: optuna.importance.get_param_importances(estudo, evaluator=avaliador)
                        for nome_conj, estudo in estudos.items()})
    imp.sort_values(imp.columns[0]).plot.barh(ax=axes[1])
    axes[1].set(title=f"{nome}: importância dos hiperparâmetros (fANOVA)", xlabel="importância")
    fig.tight_layout()
    plt.show()

#### 5.3 Random Forest

O Random Forest treina muitas árvores profundas, cada uma em uma amostra *bootstrap* dos clientes e escolhendo cada corte entre um subconjunto aleatório de variáveis, e faz a média das probabilidades. A média reduz a variância de cada árvore.
- **`max_depth`**: profundidade máxima de cada árvore.
- **`min_samples_leaf`**: mínimo de clientes em cada folha. Valores maiores = folhas mais "médias" e menos sobreajuste.
- **`max_features`**: fração das variáveis sorteadas como candidatas em cada corte. Valores menores = árvores mais diferentes entre si.
- **`max_samples`**: fração dos clientes no *bootstrap* de cada árvore.

O número de árvores fica fixo em 300: mais árvores nunca pioram o Random Forest, só deixam a média mais estável e o treino mais lento, então não é um hiperparâmetro a ser ajustado.

In [ ]:
RF = RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=SEMENTE)


def espaco_rf(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 100, log=True),
        "max_features": trial.suggest_float("max_features", 0.05, 1.0),
        "max_samples": trial.suggest_float("max_samples", 0.3, 1.0),
    }


estudos_rf = busca_optuna("RandomForest", RF, espaco_rf)

In [ ]:
mostrar_optuna(estudos_rf, "Random Forest")

In [ ]:
mostrar_melhores("RandomForest")

#### 5.4 XGBoost

No *boosting*, as árvores são treinadas em sequência: cada nova árvore corrige os erros das anteriores, e a previsão é a soma das contribuições de todas.
- **`n_estimators`** e **`learning_rate`**: número de árvores e o peso de cada uma. Os dois andam juntos: uma taxa de aprendizado menor precisa de mais árvores.
- **`max_depth`** e **`min_child_weight`**: complexidade de cada árvore (profundidade e "tamanho" mínimo de cada folha).
- **`subsample`** e **`colsample_bytree`**: fração dos clientes e das variáveis sorteadas para cada árvore (aleatoriedade, como no Random Forest).
- **`gamma`**, **`reg_lambda`** e **`reg_alpha`**: regularização. `gamma` é o ganho mínimo para fazer um corte; `reg_lambda` (L2) e `reg_alpha` (L1) penalizam os valores das folhas.

In [ ]:
XGB = XGBClassifier(tree_method="hist", n_jobs=1, random_state=SEMENTE)


def espaco_xgb(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 50, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-3, 5, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
    }


estudos_xgb = busca_optuna("XGBoost", XGB, espaco_xgb)

In [ ]:
mostrar_optuna(estudos_xgb, "XGBoost")

In [ ]:
mostrar_melhores("XGBoost")

#### 5.5 LightGBM

Também é *boosting*, mas as árvores crescem folha a folha (sempre dividindo a folha com maior ganho) em vez de nível a nível. Por isso a complexidade é controlada principalmente pelo número de folhas, e não pela profundidade.
- **`n_estimators`** e **`learning_rate`**: como no XGBoost.
- **`num_leaves`**: número máximo de folhas por árvore (com 2 folhas, cada árvore é um único corte).
- **`min_child_samples`**: mínimo de clientes em cada folha.
- **`subsample`** e **`colsample_bytree`**: fração dos clientes (sorteada a cada árvore, `subsample_freq=1`) e das variáveis.
- **`reg_lambda`** e **`reg_alpha`**: regularização L2 e L1 dos valores das folhas.

In [ ]:
LGBM = LGBMClassifier(subsample_freq=1, n_jobs=1, random_state=SEMENTE, verbose=-1)


def espaco_lgbm(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 64, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 400, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
    }


estudos_lgbm = busca_optuna("LightGBM", LGBM, espaco_lgbm)

In [ ]:
mostrar_optuna(estudos_lgbm, "LightGBM")

In [ ]:
mostrar_melhores("LightGBM")